In [1]:
import asyncio
import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from agents import Agent, Runner, WebSearchTool, function_tool, trace
from agents.model_settings import ModelSettings

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"
HOW_MANY_SEARCHES = 3

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def push(message):
    print(f"Push: {message[:100]}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message[:1000]}
    response = requests.post(pushover_url, data=payload, timeout=10)
    print("Status:", response.status_code)

In [3]:
push("Deep research setup is working")

Push: Deep research setup is working
Status: 200


In [4]:
SEARCH_INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and
produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""

search_agent = Agent(
    name="Search Agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool()],
    model=MODEL_NAME,
    model_settings=ModelSettings(tool_choice="required"),
)

In [5]:
task = "Most popular AI agent frameworks in 2026"

result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

In 2026, the most commonly cited AI agent frameworks are **LangChain/LangGraph, CrewAI, AutoGen, LlamaIndex, OpenAI Agents SDK, Google ADK, Pydantic AI, and Semantic Kernel**. Which one is “most popular” depends on the metric: one 2026 roundup says **LangChain** leads on developer survey use, package downloads, and GitHub stars, while search-trend data puts **LangGraph** ahead in branded search interest. ([drydock.si](https://drydock.si/guides/what-is-the-most-popular-ai-agent-framework?utm_source=openai))

A practical read is that **LangGraph** and **CrewAI** are especially visible in agent discussions, **AutoGen** and **LlamaIndex** remain major community options, and **OpenAI Agents SDK** and **Google ADK** are strong first-party choices. The broader takeaway from recent coverage is that there is no single universal winner; popularity varies by whether you measure surveys, downloads, stars, or search demand. ([drydock.si](https://drydock.si/guides/what-is-the-most-popular-ai-agent-framework?utm_source=openai))

In [6]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [7]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [8]:
PLANNER_INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(
    name="Planner Agent",
    instructions=PLANNER_INSTRUCTIONS,
    model=MODEL_NAME,
    output_type=WebSearchPlan,
)

In [9]:
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find current rankings, market adoption, and expert roundups of AI agent frameworks in 2026.', query='most popular AI agent frameworks 2026 ranking'), WebSearchItem(reason='Compare major frameworks and identify which ones are widely used in production today.', query='best AI agent frameworks 2026 LangGraph CrewAI AutoGen review'), WebSearchItem(reason='Capture latest mentions of newer or emerging frameworks and current ecosystem trends.', query='AI agent frameworks 2026 trends popular frameworks')])

In [10]:
for item in result.final_output.searches:
    print("Search:", item.query)
    print("Why:", item.reason)
    print()

Search: most popular AI agent frameworks 2026 ranking
Why: Find current rankings, market adoption, and expert roundups of AI agent frameworks in 2026.

Search: best AI agent frameworks 2026 LangGraph CrewAI AutoGen review
Why: Compare major frameworks and identify which ones are widely used in production today.

Search: AI agent frameworks 2026 trends popular frameworks
Why: Capture latest mentions of newer or emerging frameworks and current ecosystem trends.



In [11]:
WRITER_INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="Writer Agent",
    instructions=WRITER_INSTRUCTIONS,
    model=MODEL_NAME,
    output_type=ReportData,
)

In [12]:
fake_research = [
    "Summary one: OpenAI Agents SDK is a lightweight framework with tools, handoffs and guardrails.",
    "Summary two: LangGraph models agents as graphs with state and is popular for complex workflows.",
    "Summary three: CrewAI organises agents into role-based teams that work on tasks.",
]

writer_input = f"Original query: {task}\nSummarized search results: {fake_research}"

result = await Runner.run(writer_agent, writer_input)
report = result.final_output

print(report.short_summary)
print(report.follow_up_questions)

The most popular AI agent frameworks in 2026 appear to be OpenAI Agents SDK, LangGraph, and CrewAI. They differ mainly in how they structure agent behavior: lightweight tool-and-handoff orchestration, graph-based stateful workflows, and role-based multi-agent teams.
['How do these frameworks compare on production observability and debugging tools?', 'What are the main pricing and ecosystem differences between OpenAI Agents SDK, LangGraph, and CrewAI?', 'Which framework is best for enterprise compliance and security requirements?', 'How do these frameworks integrate with memory, vector databases, and external APIs?', 'What emerging agent frameworks could challenge these three in 2026 and beyond?']


In [13]:
display(Markdown(report.markdown_report))

# Most Popular AI Agent Frameworks in 2026

## Executive Summary

AI agent frameworks in 2026 are converging around a few dominant design patterns: lightweight orchestration, stateful workflow graphs, and role-based multi-agent collaboration. Based on the provided research, the leading frameworks are **OpenAI Agents SDK**, **LangGraph**, and **CrewAI**. Each has a distinct philosophy and is suited to different classes of agent systems.

- **OpenAI Agents SDK** emphasizes simplicity and fast development, with built-in support for tools, handoffs, and guardrails.
- **LangGraph** is designed for more complex, stateful agent workflows, using a graph-based model that gives developers fine-grained control over execution.
- **CrewAI** focuses on organizing agents into collaborative teams, where each agent has a role and works toward a shared objective.

Taken together, these frameworks reflect the maturation of the agent ecosystem. In 2026, the key differentiator is no longer whether a framework can build an agent, but how well it supports reliability, control, coordination, and production deployment.

---

## 1. What “popular” means in the 2026 agent ecosystem

Popularity in AI agent frameworks can mean several things:

1. **Adoption by developers and teams**
2. **Visibility in documentation, tutorials, and community discussions**
3. **Fit for production-grade use cases**
4. **Ability to support a wide range of agent patterns**

The research suggests that the leading frameworks are popular because they address different but important needs in agent development. No single framework dominates every use case. Instead, teams select a framework based on the tradeoff between ease of use, control, and scalability.

---

## 2. OpenAI Agents SDK

### Overview

OpenAI Agents SDK is described as a lightweight framework focused on core agent capabilities: **tools**, **handoffs**, and **guardrails**. This makes it especially attractive for teams that want to get from prototype to working agent quickly without adopting a large or complex abstraction layer.

### Core strengths

#### a. Lightweight design
The SDK is intentionally minimal. That means developers can build agents without needing to manage a heavy framework structure. This is useful for:

- Small teams
- Rapid prototyping
- Tool-using assistants
- Straightforward production workflows

#### b. Tool integration
A major feature of modern agents is the ability to call external tools. OpenAI Agents SDK supports this directly, making it easier to connect agents to APIs, databases, internal services, and search tools.

#### c. Handoffs
Handoffs enable one agent to transfer control to another when a task requires different capabilities. This is important in systems where different specialized agents handle different phases of a request.

#### d. Guardrails
Guardrails help constrain agent behavior, improving reliability and reducing unsafe or unintended outputs. In production systems, this is often one of the most important features.

### Best suited for

OpenAI Agents SDK is a strong fit for:

- Customer support assistants
- Internal productivity agents
- API-connected enterprise tools
- Quick-to-deploy agent prototypes

### Why it is popular

Its popularity likely comes from three things: a low barrier to entry, direct access to core agent capabilities, and strong alignment with production requirements. For teams already in the OpenAI ecosystem, it is a natural choice.

---

## 3. LangGraph

### Overview

LangGraph is a framework that models agents as **graphs with state**. This is a powerful approach for developers building complex workflows that require branching logic, memory, retries, conditional paths, or multi-step coordination.

### Core strengths

#### a. Graph-based control
Instead of a simple linear chain of steps, LangGraph lets developers define agent behavior as a graph. This is especially useful when workflows are not strictly sequential and may need to loop, branch, or recover from errors.

#### b. Stateful execution
State is central to LangGraph. Agents can maintain and update shared state as they move through the graph, which helps support:

- Long-running tasks
- Multi-turn interactions
- Workflow persistence
- Coordinated multi-agent logic

#### c. Flexibility for complex systems
LangGraph is well suited for systems where agent behavior must be explicit and controllable. This makes it appealing for enterprise and infrastructure-heavy applications.

### Best suited for

LangGraph is a strong fit for:

- Complex business workflows
- Multi-agent systems with branching logic
- Stateful assistants
- Agentic pipelines requiring observability and control

### Why it is popular

LangGraph’s popularity comes from its ability to handle complexity without forcing developers into simplistic patterns. As agent systems became more operationally demanding in 2026, the need for explicit state management and graph-based orchestration became increasingly important.

---

## 4. CrewAI

### Overview

CrewAI organizes agents into **role-based teams** that collaborate on tasks. This makes it one of the most intuitive frameworks for teams that want to model agent systems after human organizational structures.

### Core strengths

#### a. Role-based design
Each agent can be assigned a role, such as researcher, writer, planner, or reviewer. This creates a natural way to divide responsibilities among multiple agents.

#### b. Collaborative task execution
CrewAI is built around the idea of agents working together. That makes it useful for workflows that benefit from specialization and division of labor.

#### c. Simple conceptual model
The team-and-role abstraction is easy to understand, especially for users coming from project management or organizational thinking rather than software architecture.

### Best suited for

CrewAI is a strong fit for:

- Multi-agent content generation
- Research workflows
- Collaborative task automation
- Simulated team-based agent systems

### Why it is popular

CrewAI’s popularity likely stems from how intuitive it is. Many users find the “agents as a team” metaphor easier to grasp than more technical orchestration models. This makes it accessible to both developers and non-specialist builders.

---

## 5. Comparison of the three frameworks

### Ease of use
- **OpenAI Agents SDK**: Easiest for lightweight use cases
- **CrewAI**: Easy to conceptualize and set up for team workflows
- **LangGraph**: More complex, but powerful

### Control and customization
- **LangGraph**: Highest level of control
- **OpenAI Agents SDK**: Moderate control with simpler abstractions
- **CrewAI**: Good for role-based patterns, but less explicit than LangGraph for stateful logic

### Best for workflow complexity
- **LangGraph**: Best for complex branching and persistent state
- **CrewAI**: Best for collaborative agent teams
- **OpenAI Agents SDK**: Best for straightforward, production-ready assistants

### Production readiness
All three frameworks are relevant for production, but they differ in operational style:

- **OpenAI Agents SDK** prioritizes simplicity and safety
- **LangGraph** prioritizes orchestration depth and reliability
- **CrewAI** prioritizes human-like team structures and task delegation

---

## 6. Broader trends shaping framework popularity in 2026

### a. Reliability over novelty
Early agent systems often prioritized demonstration value. By 2026, the focus is more on reliability, observability, and bounded behavior. Frameworks that support guardrails, state, and structured execution are more attractive.

### b. Multi-agent coordination
Many real-world problems require multiple specialized agents. Frameworks that make coordination explicit—whether through handoffs, graphs, or team structures—are well positioned.

### c. Production deployment
Teams increasingly need frameworks that can handle testing, debugging, tracing, and control. Simpler demos are no longer enough.

### d. Tool-centric agent design
Agents are increasingly judged by how well they interact with external systems. Tool use remains a foundational feature across the most popular frameworks.

---

## 7. Practical guidance: which framework to choose

### Choose OpenAI Agents SDK if:
- You want a lightweight framework
- You need fast development
- Your workflow is straightforward
- You want built-in guardrails and handoffs

### Choose LangGraph if:
- Your system is stateful and complex
- You need branching workflows
- You require fine-grained orchestration
- You are building a serious multi-step or multi-agent system

### Choose CrewAI if:
- You want a team-based agent model
- Your workflow maps naturally to roles
- You are building collaborative or research-oriented systems
- You want an intuitive multi-agent abstraction

---

## 8. Conclusion

In 2026, the most popular AI agent frameworks are **OpenAI Agents SDK, LangGraph, and CrewAI**. Their popularity reflects three major approaches to agent design: lightweight orchestration, stateful graph execution, and role-based team collaboration.

There is no single winner across all use cases. Instead, the best framework depends on the problem:

- **OpenAI Agents SDK** for simplicity and safety
- **LangGraph** for complex stateful control
- **CrewAI** for collaborative multi-agent workflows

As the agent ecosystem continues to mature, these frameworks are likely to remain central because they address the core needs of production-grade AI agents: reliability, coordination, and practical deployment.


In [ ]:
## deliver the report with plain Python (no agent)


In [14]:
from datetime import datetime
from pathlib import Path

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)

In [15]:
def save_report(query, report):
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    file_path = REPORTS_DIR / f"report_{timestamp}.md"

    content = f"# {query}\n\n{report.markdown_report}\n\n## Follow-up questions\n\n"
    for question in report.follow_up_questions:
        content += f"- {question}\n"

    file_path.write_text(content, encoding="utf-8")
    return file_path


def notify(query, report, file_path):
    push(f"Research done: {query}\n\n{report.short_summary}\n\nSaved to {file_path}")

In [16]:
saved_path = save_report(task, report)
notify(task, report, saved_path)
print(saved_path)

Push: Research done: Most popular AI agent frameworks in 2026

The most popular AI agent frameworks in 202
Status: 200
reports\report_2026-10-30_01-20.md
